## **Predicting Loan Default Loss (Loss Given Default)**
# Business Context **bold text**
When a customer defaults on a loan, the lender does not necessarily lose the full outstanding balance. A portion of the exposure may be recovered through repayments, collateral liquidation, or legal recovery processes. However, the uncertainty around how much is ultimately recovered creates risk in capital planning, pricing, and portfolio management.

# **Problem Statement**
The organization currently faces the challenge of accurately estimating how much money is actually lost once a borrower has defaulted. Without reliable predictions, lenders may:

Hold excess regulatory capital
Under‑ or over‑price credit risk
Make suboptimal recovery and collection decisions

# **Proposed Solution**
This project proposes developing a Loss Given Default (LGD) prediction model that estimates the percentage of a loan exposure that is unrecoverable after a default has occurred. The model will use borrower characteristics, loan terms, credit history, and post‑default payment behavior to generate data‑driven LGD estimates.

# **Business Value**
Regulatory compliance: LGD is a core input in Basel III capital calculations, directly impacting risk‑weighted assets and capital requirements.

Capital efficiency: More accurate LGD estimates enable better alignment between true risk and capital held.

Improved risk management: Helps identify high‑loss segments and prioritize recovery strategies.

Better pricing and portfolio decisions: Supports risk‑based pricing and improves expected loss forecasting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv("/content/loans_full_schema.csv")

In [ ]:
df.head()

,emp_title,emp_length,state,homeownership,annual_income,verified_income,debt_to_income,annual_income_joint,verification_income_joint,debt_to_income_joint,...,sub_grade,issue_month,loan_status,initial_listing_status,disbursement_method,balance,paid_total,paid_principal,paid_interest,paid_late_fees
0,global config engineer,3.0,NJ,MORTGAGE,90000.0,Verified,18.01,NaN,NaN,NaN,...,C3,Mar-2018,Current,whole,Cash,27015.86,1999.33,984.14,1015.19,0.0
1,warehouse office clerk,10.0,HI,RENT,40000.0,Not Verified,5.04,NaN,NaN,NaN,...,C1,Feb-2018,Current,whole,Cash,4651.37,499.12,348.63,150.49,0.0
2,assembly,3.0,WI,RENT,40000.0,Source Verified,21.15,NaN,NaN,NaN,...,D1,Feb-2018,Current,fractional,Cash,1824.63,281.80,175.37,106.43,0.0
3,customer service,1.0,PA,RENT,30000.0,Not Verified,10.16,NaN,NaN,NaN,...,A3,Jan-2018,Current,whole,Cash,18853.26,3312.89,2746.74,566.15,0.0
4,security supervisor,10.0,CA,RENT,35000.0,Verified,57.96,57000.0,Verified,37.66,...,C3,Mar-2018,Current,whole,Cash,21430.15,2324.65,1569.85,754.80,0.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 55 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   emp_title                         9167 non-null   object 
 1   emp_length                        9183 non-null   float64
 2   state                             10000 non-null  object 
 3   homeownership                     10000 non-null  object 
 4   annual_income                     10000 non-null  float64
 5   verified_income                   10000 non-null  object 
 6   debt_to_income                    9976 non-null   float64
 7   annual_income_joint               1495 non-null   float64
 8   verification_income_joint         1455 non-null   object 
 9   debt_to_income_joint              1495 non-null   float64
 10  delinq_2y                         10000 non-null  int64  
 11  months_since_last_delinq          4342 non-null   float64
 12  earli

In [ ]:
# list(df.columns)

🧑‍💼 Borrower & Employment Information

emp_title – Job title of the borrower (e.g., Manager, Nurse).

emp_length – Length of employment (e.g., < 1 year, 5 years, 10+ years).

state – US state where the borrower resides.

homeownership – Housing situation (e.g., RENT, OWN, MORTGAGE).


💰 Income & Verification

annual_income – Individual borrower’s annual income.

verified_income – Whether income was verified (Verified, Not Verified).

debt_to_income (DTI) – Ratio of monthly debt payments to income (lower is better).

Joint Application (if applicable)

annual_income_joint – Combined income for joint applicants.

verification_income_joint – Verification status of joint income.

debt_to_income_joint – Combined DTI for joint borrowers.


📉 Credit History & Delinquencies

delinq_2y – Number of times borrower was delinquent in last 2 years.

months_since_last_delinq – Months since last delinquency.

earliest_credit_line – Date when borrower first opened a credit account.

inquiries_last_12m – Credit inquiries in the last 12 months.


🧾 Credit Accounts Overview

total_credit_lines – Total number of credit accounts.

open_credit_lines – Number of currently open credit lines.

total_credit_limit – Total available credit across all accounts.

total_credit_utilized – Total amount of credit currently used.

total_credit_utilized % (implicit) – Used credit ÷ available credit.


⚠️ Collections & Defaults

num_collections_last_12m – Collections in last 12 months.

num_historical_failed_to_pay – Historical payment failures.

months_since_90d_late – Months since last 90+ day late payment.

current_accounts_delinq – Accounts currently delinquent.

total_collection_amount_ever – Total amount ever sent to collections.


💳 Account Activity

current_installment_accounts – Installment loans currently active.

accounts_opened_24m – Accounts opened in last 24 months.

months_since_last_credit_inquiry – Months since last hard inquiry.

num_satisfactory_accounts – Accounts in good standing.


⏰ Late Payment Counts

num_accounts_120d_past_due – Accounts 120+ days late.

num_accounts_30d_past_due – Accounts 30+ days late.


🏦 Debit & Credit Card Details

num_active_debit_accounts – Active debit accounts.

total_debit_limit – Total debit limit.

num_total_cc_accounts – Total credit card accounts.

num_open_cc_accounts – Open credit card accounts.

num_cc_carrying_balance – Credit cards with outstanding balances.


🏠 Mortgage & Credit Health

num_mort_accounts – Number of mortgage accounts.

account_never_delinq_percent – % of accounts never delinquent.

tax_liens – Number of tax liens.

public_record_bankrupt – Whether borrower has declared bankruptcy.


📄 Loan Application Details

loan_purpose – Reason for the loan (e.g., debt consolidation, car).

application_type – Individual or Joint.

loan_amount – Amount borrowed.

term – Loan duration (e.g., 36 or 60 months).

interest_rate – Interest rate charged.

installment – Monthly payment amount.

grade – Credit quality grade (A = best, G = worst).

sub_grade – More granular grade (e.g., A1, B3).

issue_month – Month the loan was issued.


🧾 Loan Status & Administration

loan_status – Current status (Fully Paid, Charged Off, Late, etc.).

initial_listing_status – Initial funding status.

disbursement_method – How funds were paid to borrower.


💵 Payment & Balance Tracking

balance – Current outstanding loan balance.

paid_total – Total amount paid so far.

paid_principal – Amount of principal paid.

paid_interest – Interest paid so far.

paid_late_fees – Late fees paid.

